# 🧪 LLM Bilgi Damıtma (Knowledge Distillation) — Colab Pro+

**Öğretmen:** `Qwen/Qwen2.5-7B-Instruct` → **Öğrenci:** `Qwen/Qwen2.5-0.5B-Instruct`

Öğrenci model, öğretmenin her token için ürettiği olasılık dağılımını (logit'leri) taklit etmeyi öğrenir.
Kayıp fonksiyonu:

$$\mathcal{L} = \alpha \cdot T^2 \cdot \mathrm{KL}\big(p_{\text{öğretmen}}^{(T)} \,\|\, p_{\text{öğrenci}}^{(T)}\big) + (1-\alpha) \cdot \mathrm{CE}(y, p_{\text{öğrenci}})$$

Kayıp yalnızca **cevap token'larında** hesaplanır (prompt maskelenir).

**Önerilen çalışma zamanı:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU` + **Yüksek RAM**.
L4 / T4 gelirse ayarlar otomatik olarak küçülür (öğretmen 4-bit yüklenir).

> Her kod satırının yanında / üstünde `#` ile başlayan Türkçe açıklama vardır.

## 1) GPU kontrolü

In [ ]:
!nvidia-smi                                   # Colab'ın verdiği GPU'yu, sürücüyü ve bellek durumunu gösterir

import torch                                  # PyTorch: tensör işlemleri ve sinir ağları kütüphanesi

print("CUDA:", torch.cuda.is_available())     # GPU'ya erişilebiliyor mu? (True olmalı)

if torch.cuda.is_available():                 # GPU varsa bilgilerini oku
    GPU_NAME = torch.cuda.get_device_name(0)  # 0 numaralı GPU'nun adı (ör. "NVIDIA A100-SXM4-40GB")
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9  # Toplam GPU belleği (bayt → GB)
    print(f"GPU: {GPU_NAME}  |  Bellek: {GPU_MEM_GB:.1f} GB")            # Ad ve belleği ekrana yaz
    print("bf16 destekli:", torch.cuda.is_bf16_supported())              # A100/L4 bf16 destekler, T4 desteklemez
else:                                         # GPU yoksa
    raise RuntimeError("GPU yok! Çalışma zamanı türünü GPU (tercihen A100) yapın.")  # çalışmayı durdur

Wed Nov  4 12:03:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2) Kurulum

In [ ]:
# -q          : sessiz kurulum (daha az çıktı)
# -U          : zaten kuruluysa güncelle
# transformers: Hugging Face model yükleme / üretme kütüphanesi
# datasets    : Hugging Face veri seti indirme ve işleme
# accelerate  : modelleri GPU'ya verimli yerleştirme (device_map) için gerekli
# bitsandbytes: 4-bit / 8-bit nicemleme (küçük GPU'da öğretmeni sığdırmak için)
# sentencepiece: bazı tokenizer'ların ihtiyaç duyduğu alt-kelime kütüphanesi
!pip install -q -U "transformers>=4.45" "datasets>=2.20" "accelerate>=0.34" bitsandbytes sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


## 3) (İsteğe bağlı) Google Drive — checkpoint'ler oturum kapansa da kalsın

In [ ]:
USE_DRIVE = True                                   # Drive kullanmak istemiyorsanız False yapın

import os                                          # Dosya/klasör işlemleri için standart kütüphane

if USE_DRIVE:                                      # Drive kullanılacaksa
    from google.colab import drive                 # Colab'ın Drive bağlama modülü
    drive.mount('/content/drive')                  # Drive'ı bağla (ilk seferde izin penceresi açılır)
    OUTPUT_DIR = "/content/drive/MyDrive/distil_qwen_0.5b"  # Modeller Drive'da bu klasöre kaydedilecek
else:                                              # Drive kullanılmayacaksa
    OUTPUT_DIR = "/content/distil_qwen_0.5b"       # Geçici Colab diskine kaydet (oturum kapanınca silinir!)

os.makedirs(OUTPUT_DIR, exist_ok=True)             # Klasörü oluştur; zaten varsa hata verme
print("Çıktı klasörü:", OUTPUT_DIR)                # Nereye kaydedileceğini göster

Mounted at /content/drive
Çıktı klasörü: /content/drive/MyDrive/distil_qwen_0.5b


## 4) Ayarlar
GPU'ya göre otomatik seçilir; istediğiniz değeri elle değiştirebilirsiniz.

In [ ]:
from dataclasses import dataclass, asdict      # dataclass: ayarları düzenli bir sınıfta tutmak için

BIG_GPU = GPU_MEM_GB >= 38                     # 38 GB+ bellek varsa "büyük GPU" (A100 40/80GB, H100)
BF16 = torch.cuda.is_bf16_supported()          # bf16 destekleniyorsa onu kullan; T4'te False → fp16

@dataclass                                     # Aşağıdaki sınıfı otomatik __init__ ile veri sınıfı yap
class CFG:
    # ---------------- Modeller ----------------
    teacher_id: str = "Qwen/Qwen2.5-7B-Instruct"    # Büyük, akıllı öğretmen model (7 milyar parametre)
    student_id: str = "Qwen/Qwen2.5-0.5B-Instruct"  # Küçük öğrenci model (0.5 milyar parametre)
    teacher_4bit: bool = not BIG_GPU                # Küçük GPU'da öğretmeni 4-bit yükle (≈5 GB yerine ≈15 GB)

    # ---------------- Veri ----------------
    dataset_id: str = "yahma/alpaca-cleaned"   # instruction / input / output sütunlu talimat veri seti
    max_train_samples: int = 20000             # Eğitimde kullanılacak en fazla örnek sayısı
    max_eval_samples: int = 300                # Değerlendirme (eval) için ayrılacak örnek sayısı
    max_len: int = 512                         # Bir örneğin en fazla token uzunluğu (fazlası kesilir)

    # ---------------- Eğitim ----------------
    epochs: int = 1                            # Veri setinin üzerinden kaç kez geçileceği
    batch_size: int = 8 if BIG_GPU else 2      # Bir ileri geçişte işlenecek örnek sayısı
    grad_accum: int = 4 if BIG_GPU else 16     # Kaç mini-batch'in gradyanı toplanıp tek adım atılacak (efektif batch = 32)
    lr: float = 5e-5                           # Öğrenme oranı (learning rate)
    weight_decay: float = 0.01                 # Ağırlıkların aşırı büyümesini engelleyen düzenlileştirme
    warmup_ratio: float = 0.03                 # Toplam adımların %3'ünde LR sıfırdan yavaşça yükselir
    max_grad_norm: float = 1.0                 # Gradyan kırpma: patlayan gradyanları sınırlar

    # ---------------- Damıtma ----------------
    temperature: float = 2.0                   # Sıcaklık: dağılımları yumuşatır, "karanlık bilgi"yi açığa çıkarır
    alpha: float = 0.7                         # KL kaybının ağırlığı (CE ağırlığı = 1 - alpha = 0.3)

    # ---------------- Kayıt / log ----------------
    log_every: int = 10                        # Her 10 optimizer adımında bir kaybı yazdır
    eval_every: int = 200                      # Her 200 adımda bir eval setinde ölç
    save_every: int = 500                      # Her 500 adımda bir checkpoint kaydet
    seed: int = 42                             # Rastgelelik tohumu (tekrarlanabilirlik için)

cfg = CFG()                                                 # Ayar nesnesini oluştur
DTYPE = torch.bfloat16 if BF16 else torch.float16           # Hesaplamada kullanılacak düşük hassasiyetli veri tipi

for k, v in asdict(cfg).items():                            # Tüm ayarları sözlüğe çevirip tek tek dolaş
    print(f"{k:>18}: {v}")                                  # Sağa hizalı şekilde ekrana yaz

        teacher_id: Qwen/Qwen2.5-7B-Instruct
        student_id: Qwen/Qwen2.5-0.5B-Instruct
      teacher_4bit: True
        dataset_id: yahma/alpaca-cleaned
 max_train_samples: 20000
  max_eval_samples: 300
           max_len: 512
            epochs: 1
        batch_size: 2
        grad_accum: 16
                lr: 5e-05
      weight_decay: 0.01
      warmup_ratio: 0.03
     max_grad_norm: 1.0
       temperature: 2.0
             alpha: 0.7
         log_every: 10
        eval_every: 200
        save_every: 500
              seed: 42


## 5) Modelleri yükle

In [ ]:
import gc, math, random, time                  # gc: bellek temizliği, math: matematik, random: rastgelelik, time: süre ölçümü
import numpy as np                             # NumPy: sayısal hesaplama
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig  # Tokenizer, model ve 4-bit ayar sınıfları

random.seed(cfg.seed)                          # Python'un rastgele sayı üretecini sabitle
np.random.seed(cfg.seed)                       # NumPy'ın rastgele sayı üretecini sabitle
torch.manual_seed(cfg.seed)                    # PyTorch'un rastgele sayı üretecini sabitle
device = "cuda"                                # Tüm hesaplamalar GPU'da yapılacak

# ---------------- Tokenizer ----------------
tokenizer = AutoTokenizer.from_pretrained(cfg.student_id)  # Metni token ID'lerine çeviren tokenizer'ı indir
if tokenizer.pad_token is None:                # Doldurma (padding) token'ı tanımlı değilse
    tokenizer.pad_token = tokenizer.eos_token  # cümle sonu token'ını padding olarak kullan
tokenizer.padding_side = "right"               # Eğitimde dolgu sağa eklenir (sol tarafta gerçek metin)

# ---------------- Öğretmen (dondurulmuş) ----------------
t_kwargs = dict(torch_dtype=DTYPE,             # Öğretmen ağırlıklarını bf16/fp16 yükle (yarı bellek)
                device_map={"": 0})            # Tüm modeli 0 numaralı GPU'ya yerleştir
if cfg.teacher_4bit:                           # Küçük GPU modundaysak
    t_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,                     # Ağırlıkları 4-bit'e sıkıştır
        bnb_4bit_quant_type="nf4",             # NF4: LLM ağırlıkları için optimize edilmiş 4-bit format
        bnb_4bit_compute_dtype=DTYPE,          # Matris çarpımları bf16/fp16 ile yapılsın
        bnb_4bit_use_double_quant=True)        # Nicemleme sabitlerini de sıkıştır (ek bellek kazancı)
teacher = AutoModelForCausalLM.from_pretrained(cfg.teacher_id, **t_kwargs)  # Öğretmeni indir ve yükle
teacher.eval()                                 # Değerlendirme moduna al (dropout vb. kapanır)
for p in teacher.parameters():                 # Öğretmenin tüm ağırlıklarını dolaş
    p.requires_grad_(False)                    # Gradyan hesaplanmasın → öğretmen asla güncellenmez

# ---------------- Öğrenci (eğitilecek) ----------------
student = AutoModelForCausalLM.from_pretrained(cfg.student_id,
                                               torch_dtype=torch.float32  # Ağırlıklar fp32 tutulur (kararlı eğitim)
                                               ).to(device)              # GPU'ya taşı
student.gradient_checkpointing_enable()        # Ara aktivasyonları saklamak yerine yeniden hesapla → bellek tasarrufu
student.config.use_cache = False               # Eğitimde KV-cache gereksiz ve gradient checkpointing ile uyumsuz

# Qwen 0.5B ve 7B'nin çıkış katmanı boyutları farklı (151936 vs 152064).
# Gerçek token'lar her ikisinde de aynı; fazlası sadece dolgu. Bu yüzden ortak (küçük) kısmı kullanıyoruz.
VOCAB = min(teacher.config.vocab_size, student.config.vocab_size, len(tokenizer))  # Ortak sözlük boyutu
print("Ortak vocab:", VOCAB)                   # Ekrana yaz

n = lambda m: sum(p.numel() for p in m.parameters()) / 1e6   # Modelin parametre sayısını milyon cinsinden hesaplayan fonksiyon
print(f"Öğretmen: {n(teacher):.0f}M param | Öğrenci: {n(student):.0f}M param")  # İki modelin boyutunu yaz
print(f"GPU bellek kullanımı: {torch.cuda.memory_allocated()/1e9:.1f} GB")     # Şu an GPU'da ne kadar bellek dolu

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

... (7 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ortak vocab: 151665
Öğretmen: 4353M param | Öğrenci: 494M param
GPU bellek kullanımı: 7.5 GB


## 6) Veri seti — sohbet şablonu + prompt maskeleme

In [ ]:
from datasets import load_dataset                          # Hugging Face veri seti yükleyici

raw = load_dataset(cfg.dataset_id, split="train")          # Veri setinin eğitim bölümünü indir
raw = raw.shuffle(seed=cfg.seed)                           # Örnekleri karıştır (sabit tohumla)
raw = raw.select(range(min(len(raw),                       # Veri setinin tamamından fazlasını isteme
                           cfg.max_train_samples + cfg.max_eval_samples)))  # Sadece gereken kadar örnek al

def build_prompt(ex):                                      # Bir örnekten kullanıcı mesajını oluşturan fonksiyon
    if ex.get("input"):                                    # Ek girdi (input) alanı doluysa
        user = f"{ex['instruction']}\n\n{ex['input']}"     # talimat + boş satır + girdi
    else:                                                  # Ek girdi yoksa
        user = ex["instruction"]                           # sadece talimat
    return [{"role": "user", "content": user}]             # Sohbet formatında mesaj listesi döndür

def tokenize(ex):                                          # Bir örneği token ID'lerine ve etiketlere çeviren fonksiyon
    msgs = build_prompt(ex)                                # Kullanıcı mesajını oluştur
    prompt_text = tokenizer.apply_chat_template(           # Modelin sohbet şablonunu uygula
        msgs, tokenize=False,                              # metin olarak döndür (henüz token değil)
        add_generation_prompt=True)                        # sonuna "<|im_start|>assistant" ekle → cevap buradan başlar
    full_text = prompt_text + ex["output"] + tokenizer.eos_token   # Tam metin = prompt + doğru cevap + bitiş token'ı
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]  # Sadece prompt'un token'ları
    ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]           # Tam metnin token'ları
    ids = ids[: cfg.max_len]                               # Çok uzunsa max_len'de kes
    n_prompt = min(len(prompt_ids), len(ids))              # Prompt kısmının uzunluğu (kesilmişse daha kısa)
    labels = [-100] * n_prompt + ids[n_prompt:]            # Prompt token'ları -100 → kayıpta yok sayılır; cevap token'ları etiket
    return {"input_ids": ids, "labels": labels}            # Model girdisi ve etiketleri döndür

ds = raw.map(tokenize,                                     # tokenize fonksiyonunu her örneğe uygula
             remove_columns=raw.column_names,              # Eski metin sütunlarını sil, sadece ID'ler kalsın
             num_proc=4)                                   # 4 işlemci çekirdeğiyle paralel çalış
ds = ds.filter(lambda x: any(l != -100 for l in x["labels"]))  # Cevabı tamamen kesilmiş (öğrenilecek token'ı olmayan) örnekleri at

n_eval = min(cfg.max_eval_samples, len(ds) // 10)          # Eval örnek sayısı (verinin en fazla %10'u)
eval_ds = ds.select(range(n_eval))                         # İlk n_eval örnek → değerlendirme seti
train_ds = ds.select(range(n_eval, len(ds)))               # Geri kalanı → eğitim seti
print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")   # Boyutları yaz

def collate(batch):                                        # Farklı uzunluktaki örnekleri tek tensörde birleştiren fonksiyon
    L = max(len(b["input_ids"]) for b in batch)            # Batch'teki en uzun örneğin uzunluğu
    B = len(batch)                                         # Batch'teki örnek sayısı
    ids  = torch.full((B, L), tokenizer.pad_token_id, dtype=torch.long)  # Hepsini pad token'ı ile doldurulmuş matris
    lab  = torch.full((B, L), -100, dtype=torch.long)      # Etiket matrisi; dolgu kısımları -100 (yok sayılır)
    mask = torch.zeros((B, L), dtype=torch.long)           # Dikkat maskesi: 1 = gerçek token, 0 = dolgu
    for i, b in enumerate(batch):                          # Her örneği sırayla
        k = len(b["input_ids"])                            # bu örneğin gerçek uzunluğu
        ids[i, :k]  = torch.tensor(b["input_ids"])         # gerçek token'ları matrisin başına yaz
        lab[i, :k]  = torch.tensor(b["labels"])            # etiketleri yaz
        mask[i, :k] = 1                                    # gerçek token'ların maskesini 1 yap
    return {"input_ids": ids, "labels": lab, "attention_mask": mask}  # Modelin beklediği sözlüğü döndür

from torch.utils.data import DataLoader                    # Veriyi batch'ler halinde sunan PyTorch yardımcısı

train_loader = DataLoader(train_ds,
                          batch_size=cfg.batch_size,       # Her batch'te kaç örnek
                          shuffle=True,                    # Her epoch'ta sırayı karıştır
                          collate_fn=collate,              # Batch'i yukarıdaki fonksiyonla oluştur
                          num_workers=2,                   # Veriyi 2 arka plan işçisiyle hazırla
                          pin_memory=True,                 # CPU→GPU kopyalamayı hızlandır
                          drop_last=True)                  # Eksik kalan son batch'i atla
eval_loader = DataLoader(eval_ds,
                         batch_size=cfg.batch_size,        # Eval batch boyutu
                         shuffle=False,                    # Eval'de karıştırmaya gerek yok
                         collate_fn=collate)               # Aynı birleştirme fonksiyonu

ex = train_ds[0]                                           # İlk eğitim örneğini al
answer_ids = [t for t in ex["labels"] if t != -100]        # Sadece cevap (öğrenilecek) token'larını seç
print("Örnek (cevap kısmı):", tokenizer.decode(answer_ids)[:300])  # Metne geri çevirip ilk 300 karakteri göster

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/20300 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20300 [00:00<?, ? examples/s]

Train: 19998 | Eval: 300
Örnek (cevap kısmı): An armadillo is a small to medium-sized mammal with a leathery armor shell, that belongs to the family Dasypodidae. They are found predominantly in South and Central America, with one species, the nine-banded armadillo, expanding northwards into the United States. Armadillos have sharp claws for dig


## 7) Damıtma kaybı

In [ ]:
import torch.nn.functional as F                           # Softmax, KL, cross-entropy gibi fonksiyonlar

def distill_loss(s_logits, t_logits, labels, T=cfg.temperature, alpha=cfg.alpha):
    # s_logits: öğrencinin çıktısı  [batch, uzunluk, vocab]
    # t_logits: öğretmenin çıktısı  [batch, uzunluk, vocab]
    # labels  : doğru token ID'leri [batch, uzunluk] (-100 = yok say)

    s = s_logits[:, :-1, :VOCAB].float()    # Öğrenci: son pozisyonu at (sonrası yok), ortak vocab'a kırp, fp32'ye çevir
    t = t_logits[:, :-1, :VOCAB].float()    # Öğretmen: aynı işlemler
    y = labels[:, 1:]                       # Etiketleri bir sola kaydır: i. pozisyon (i+1). token'ı tahmin eder
    m = (y != -100)                         # Maske: sadece cevap token'ları True

    s_m = s[m]                              # Öğrenci logit'lerinden sadece cevap pozisyonlarını al → [N, vocab]
    t_m = t[m]                              # Öğretmen logit'lerinden aynı pozisyonları al → [N, vocab]

    s_logp = F.log_softmax(s_m / T, dim=-1) # Öğrenci: sıcaklıkla yumuşatılmış log-olasılıklar
    t_logp = F.log_softmax(t_m / T, dim=-1) # Öğretmen: sıcaklıkla yumuşatılmış log-olasılıklar

    kl = F.kl_div(s_logp, t_logp,           # KL(öğretmen || öğrenci): öğrenci dağılımı öğretmeninkine ne kadar uzak
                  log_target=True,          # hedef (öğretmen) de log formatında verildi
                  reduction="batchmean")    # token başına ortalama
    kl = kl * (T * T)                       # T² ile çarp: sıcaklık gradyanları küçülttüğü için ölçeği geri getirir

    ce = F.cross_entropy(s_m, y[m])         # Klasik kayıp: öğrencinin doğru token'ı ne kadar iyi tahmin ettiği

    loss = alpha * kl + (1 - alpha) * ce    # İkisinin ağırlıklı toplamı = nihai kayıp
    return loss, kl.detach(), ce.detach()   # Kayıp + log için KL ve CE (detach: gradyan grafiğinden ayır)

## 8) Eğitim döngüsü

In [ ]:
from transformers import get_cosine_schedule_with_warmup  # Isınma + kosinüs azalan öğrenme oranı planlayıcısı
from tqdm.auto import tqdm                                # İlerleme çubuğu

opt = torch.optim.AdamW(student.parameters(),             # AdamW optimizer, sadece öğrencinin ağırlıklarını günceller
                        lr=cfg.lr,                        # öğrenme oranı
                        weight_decay=cfg.weight_decay,    # ağırlık azaltma
                        betas=(0.9, 0.95),                # LLM eğitiminde yaygın momentum değerleri
                        fused=True)                       # GPU'da birleşik (daha hızlı) çekirdek kullan

steps_per_epoch = math.ceil(len(train_loader) / cfg.grad_accum)   # Bir epoch'taki optimizer adımı sayısı
total_steps = steps_per_epoch * cfg.epochs                        # Toplam optimizer adımı
sched = get_cosine_schedule_with_warmup(opt,
                                        int(total_steps * cfg.warmup_ratio),  # Isınma adımı sayısı
                                        total_steps)                          # Toplam adım (sonunda LR → 0)
scaler = torch.amp.GradScaler("cuda", enabled=not BF16)  # fp16'da küçük gradyanların sıfırlanmasını önler; bf16'da gerekmez
print("Toplam optimizer adımı:", total_steps)

@torch.no_grad()                                          # Bu fonksiyonda gradyan hesaplanmaz (hızlı + az bellek)
def evaluate():
    student.eval()                                        # Öğrenciyi değerlendirme moduna al
    tot, tkl, tce, k = 0., 0., 0., 0                      # Toplam kayıp, KL, CE ve batch sayacı
    for b in eval_loader:                                 # Eval setindeki her batch için
        b = {kk: v.to(device) for kk, v in b.items()}     # Tensörleri GPU'ya taşı
        with torch.autocast("cuda", dtype=DTYPE):         # Karma hassasiyet (bf16/fp16) ile hızlı hesap
            t_out = teacher(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits  # Öğretmen tahmini
            s_out = student(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits  # Öğrenci tahmini
        l, kl, ce = distill_loss(s_out, t_out, b["labels"])  # Kayıpları hesapla
        tot += l.item(); tkl += kl.item(); tce += ce.item(); k += 1  # Topla
    student.train()                                       # Öğrenciyi tekrar eğitim moduna al
    return tot / k, tkl / k, tce / k, math.exp(tce / k)   # Ortalama kayıp, KL, CE ve perplexity (= e^CE)

def save(tag):                                            # Öğrenciyi diske kaydeden fonksiyon
    path = f"{OUTPUT_DIR}/{tag}"                          # Kayıt klasörü, ör. .../checkpoint-500
    student.save_pretrained(path, safe_serialization=True)  # Ağırlıkları .safetensors formatında kaydet
    tokenizer.save_pretrained(path)                       # Tokenizer'ı da yanına kaydet (tek başına yüklenebilsin)
    print(f"💾 Kaydedildi: {path}")

history = {"step": [], "loss": [], "kl": [], "ce": [],   # Grafikler için eğitim kayıtları
           "eval_step": [], "eval_loss": [], "eval_ppl": []}  # ve eval kayıtları

base_eval = evaluate()                                    # Eğitimden ÖNCE öğrencinin durumunu ölç (karşılaştırma için)
print(f"Başlangıç  eval_loss={base_eval[0]:.4f}  KL={base_eval[1]:.4f}  "
      f"CE={base_eval[2]:.4f}  PPL={base_eval[3]:.2f}")

student.train()                                           # Eğitim modunu aç
step, micro = 0, 0                                        # step: optimizer adımı, micro: işlenen mini-batch sayısı
run_l = run_kl = run_ce = 0.                              # Log aralığı boyunca toplanan kayıplar
t0 = time.time()                                          # Başlangıç zamanı
pbar = tqdm(total=total_steps, desc="Damıtma")            # İlerleme çubuğu

for epoch in range(cfg.epochs):                           # Her epoch için
    for batch in train_loader:                            # Her mini-batch için
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}  # Batch'i GPU'ya taşı

        with torch.no_grad(), torch.autocast("cuda", dtype=DTYPE):  # Öğretmen: gradyansız + karma hassasiyet
            t_logits = teacher(input_ids=batch["input_ids"],
                               attention_mask=batch["attention_mask"]).logits  # Öğretmenin "yumuşak hedefleri"

        with torch.autocast("cuda", dtype=DTYPE):         # Öğrenci: gradyanlı + karma hassasiyet
            s_logits = student(input_ids=batch["input_ids"],
                               attention_mask=batch["attention_mask"]).logits  # Öğrencinin tahminleri

        loss, kl, ce = distill_loss(s_logits, t_logits, batch["labels"])  # Damıtma kaybını hesapla
        scaler.scale(loss / cfg.grad_accum).backward()    # Kaybı birikim sayısına böl ve gradyanları hesapla (birikir)
        del t_logits, s_logits                            # Büyük logit tensörlerini hemen bellekten sil

        run_l += loss.item(); run_kl += kl.item(); run_ce += ce.item()  # Log için kayıpları topla
        micro += 1                                        # Mini-batch sayacını artır
        if micro % cfg.grad_accum != 0:                   # Yeterli gradyan birikmediyse
            continue                                      # optimizer adımı atmadan sonraki batch'e geç

        scaler.unscale_(opt)                              # (fp16 ise) gradyanları gerçek ölçeğe döndür
        torch.nn.utils.clip_grad_norm_(student.parameters(), cfg.max_grad_norm)  # Gradyan normunu 1.0 ile sınırla
        scaler.step(opt)                                  # Ağırlıkları güncelle
        scaler.update()                                   # Ölçek faktörünü ayarla (fp16 için)
        opt.zero_grad(set_to_none=True)                   # Biriken gradyanları sıfırla
        sched.step()                                      # Öğrenme oranını planlayıcıya göre güncelle
        step += 1; pbar.update(1)                         # Adım sayacını ve ilerleme çubuğunu ilerlet

        if step % cfg.log_every == 0:                     # Log zamanı geldiyse
            d = cfg.log_every * cfg.grad_accum            # Bu aralıkta işlenen mini-batch sayısı
            history["step"].append(step)                  # Adımı kaydet
            history["loss"].append(run_l / d)             # Ortalama toplam kayıp
            history["kl"].append(run_kl / d)              # Ortalama KL
            history["ce"].append(run_ce / d)              # Ortalama CE
            pbar.set_postfix(loss=f"{run_l/d:.3f}", kl=f"{run_kl/d:.3f}",
                             ce=f"{run_ce/d:.3f}", lr=f"{sched.get_last_lr()[0]:.1e}")  # Çubukta göster
            run_l = run_kl = run_ce = 0.                  # Sayaçları sıfırla

        if step % cfg.eval_every == 0:                    # Eval zamanı geldiyse
            el, ekl, ece, ppl = evaluate()                # Eval setinde ölç
            history["eval_step"].append(step)             # Kaydet
            history["eval_loss"].append(el)
            history["eval_ppl"].append(ppl)
            print(f"\n[adım {step}] eval_loss={el:.4f} KL={ekl:.4f} CE={ece:.4f} PPL={ppl:.2f}")

        if step % cfg.save_every == 0:                    # Kayıt zamanı geldiyse
            save(f"checkpoint-{step}")                    # Checkpoint kaydet

pbar.close()                                              # İlerleme çubuğunu kapat
print(f"Eğitim bitti: {(time.time()-t0)/60:.1f} dk")      # Toplam süreyi dakika olarak yaz
save("final")                                             # Son modeli "final" klasörüne kaydet

Toplam optimizer adımı: 625


## 9) Eğri grafikleri

In [ ]:
import matplotlib.pyplot as plt                           # Grafik çizim kütüphanesi

fig, ax = plt.subplots(1, 2, figsize=(13, 4))             # Yan yana 2 grafik alanı oluştur

ax[0].plot(history["step"], history["loss"], label="toplam")  # Sol grafik: toplam eğitim kaybı
ax[0].plot(history["step"], history["kl"], label="KL")        # KL kaybı (öğretmeni taklit)
ax[0].plot(history["step"], history["ce"], label="CE")        # CE kaybı (doğru cevabı bilme)
ax[0].set_title("Eğitim kaybı")                           # Başlık
ax[0].set_xlabel("adım")                                  # X ekseni etiketi
ax[0].legend()                                            # Çizgi açıklamalarını göster

ax[1].plot([0] + history["eval_step"],                    # Sağ grafik: 0. adım (eğitim öncesi) + eval adımları
           [base_eval[3]] + history["eval_ppl"],          # Başlangıç perplexity'si + sonraki ölçümler
           marker="o")                                    # Noktaları işaretle
ax[1].set_title("Eval perplexity (düşük = iyi)")          # Başlık
ax[1].set_xlabel("adım")                                  # X ekseni etiketi

plt.tight_layout()                                        # Grafikler üst üste binmesin
plt.show()                                                # Göster

## 10) Karşılaştırma: orijinal öğrenci vs damıtılmış öğrenci vs öğretmen

In [ ]:
student.eval()                                            # Damıtılmış öğrenciyi değerlendirme moduna al
student.config.use_cache = True                           # Metin üretirken KV-cache açık olsun (çok daha hızlı)

base_student = AutoModelForCausalLM.from_pretrained(      # Karşılaştırma için eğitilmemiş orijinal 0.5B modeli yükle
    cfg.student_id, torch_dtype=DTYPE).to(device).eval()

@torch.no_grad()                                          # Üretimde gradyan gerekmez
def generate(model, prompt, max_new_tokens=200):          # Bir modelle bir soruya cevap üreten fonksiyon
    text = tokenizer.apply_chat_template(                 # Soruyu sohbet şablonuna yerleştir
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to(device) # Token'lara çevir ve GPU'ya taşı
    with torch.autocast("cuda", dtype=DTYPE):             # Karma hassasiyet
        out = model.generate(**enc,
                             max_new_tokens=max_new_tokens,      # En fazla 200 yeni token üret
                             do_sample=False,                    # Açgözlü (greedy) çözümleme → tekrarlanabilir sonuç
                             pad_token_id=tokenizer.pad_token_id)
    new_tokens = out[0, enc["input_ids"].shape[1]:]       # Sadece yeni üretilen kısmı al (prompt'u at)
    return tokenizer.decode(new_tokens, skip_special_tokens=True)  # Metne çevir

prompts = [                                               # Test soruları (farklı beceriler)
    "Explain the difference between supervised and unsupervised learning in 3 sentences.",  # Açıklama
    "Write a Python function that checks whether a string is a palindrome.",                # Kod
    "A train travels 120 km in 1.5 hours. What is its average speed? Show your steps.",     # Matematik
    "Yapay zekâ nedir? Kısaca açıkla.",                                                     # Türkçe
]

models = [("Orijinal 0.5B", base_student),                # Karşılaştırılacak üç model
          ("Damıtılmış 0.5B", student),
          ("Öğretmen 7B", teacher)]

for p in prompts:                                         # Her soru için
    print("=" * 100, f"\n📝 {p}\n")                       # Ayraç ve soruyu yaz
    for name, m in models:                                # Her model için
        print(f"--- {name} ---\n{generate(m, p).strip()}\n")  # Modelin cevabını yaz

del base_student                                          # Orijinal modeli sil
gc.collect()                                              # Python çöp toplayıcısını çalıştır
torch.cuda.empty_cache()                                  # GPU'daki boş belleği serbest bırak

## 11) (İsteğe bağlı) Hugging Face Hub'a yükle

In [ ]:
PUSH_TO_HUB = False                                       # Yüklemek için True yapın
HUB_REPO = "kullanici-adiniz/qwen2.5-0.5b-distilled"      # Hugging Face'teki depo adı (kullanıcı-adı/model-adı)

if PUSH_TO_HUB:                                           # Yükleme açıksa
    from huggingface_hub import login                     # HF giriş fonksiyonu
    login()                                               # Erişim token'ı sorar (huggingface.co/settings/tokens)
    student.to(DTYPE).push_to_hub(HUB_REPO, private=True) # Modeli yarı hassasiyette (daha küçük dosya) özel depoya yükle
    tokenizer.push_to_hub(HUB_REPO, private=True)         # Tokenizer'ı da yükle
    print("Yüklendi:", HUB_REPO)

## 💡 İpuçları
- **Daha iyi sonuç:** `max_train_samples`'ı artırın (50k–100k) ve `epochs=2` deneyin. A100'de 20k örnek ≈ 1–1,5 saat sürer.
- **Bellek hatası (OOM):** `batch_size`'ı düşürüp `grad_accum`'u aynı oranda artırın, ya da `max_len=384` yapın.
- **Türkçe odak:** `dataset_id`'yi Türkçe bir instruction veri setiyle değiştirin; `tokenize` fonksiyonundaki sütun adlarını uyarlayın.
- **Farklı model çifti:** Öğretmen ve öğrencinin **aynı tokenizer'ı** kullanması gerekir (ör. Llama-3.1-8B → Llama-3.2-1B).
- **alpha / temperature:** `alpha` 1'e yaklaştıkça öğrenci öğretmeni daha çok taklit eder; `T=1–4` aralığı tipiktir.